# UlamAI Prover Tutorial with Examples

This notebook mirrors `examples/UlamAI_Prover_Tutorial.md` and demonstrates:
1. Lean proving smoke test
2. Informal proving to `.tex`
3. Formalization from `.tex`
4. Artifacts and resume paths


## 0) Setup Repo + Install

This clones the repo and installs the package in editable mode.

In [ ]:
%%bash
set -e
if [ ! -d ulamai ]; then
  git clone https://github.com/ulamai/ulamai.git
fi
cd ulamai
python3 -m pip install -q -e .
python3 -m ulam --help >/dev/null
echo "Setup complete."


## 1) Configure a No-Key Demo Mode

The cell below writes a local config with `llm_provider=mock` so the tutorial runs without external API keys.

For real results, later switch provider in `.ulam/config.json` (OpenAI/Claude/Gemini/Ollama/Codex CLI).

In [ ]:
%%bash
set -e
cd ulamai
mkdir -p .ulam
cat > .ulam/config.json <<'JSON'
{
  "llm_provider": "mock",
  "prove": {
    "mode": "tactic",
    "output_format": "lean",
    "solver": "script",
    "autop": true,
    "k": 1,
    "tex_out_dir": "proofs",
    "tex_rounds": 2,
    "tex_judge_repairs": 1,
    "tex_worker_drafts": 1,
    "tex_replan_passes": 1,
    "tex_artifacts_dir": "runs/prove_tex",
    "llm_rounds": 2,
    "llm_cycle_patience": 1,
    "llm_allow_helper_lemmas": true,
    "llm_edit_scope": "full",
    "llm_typecheck_backend": "cli",
    "search_lean_backend": "dojo",
    "lemma_max": 60,
    "lemma_depth": 60,
    "allow_axioms": true,
    "typecheck_timeout_s": 30,
    "retriever_k": 8,
    "retriever_source": "local",
    "retriever_build": "auto",
    "retriever_index": ""
  },
  "formalize": {
    "proof_backend": "llm",
    "lean_backend": "dojo",
    "max_rounds": 2,
    "max_repairs": 2,
    "max_equivalence_repairs": 1,
    "max_proof_rounds": 1,
    "proof_repair": 1,
    "typecheck_timeout_s": 30,
    "llm_check": false,
    "llm_check_timing": "end",
    "llm_check_repairs": 0
  },
  "segmentation": {
    "chunk_words": 1000
  },
  "lean": {
    "project": "",
    "imports": [],
    "dojo_timeout_s": 120
  },
  "policy": {
    "proof_profile": "normal"
  }
}
JSON
echo "Wrote .ulam/config.json"


In [ ]:
%%bash
cd ulamai
ls -1 examples


## 2) Lean Proving Smoke Test

In [ ]:
%%bash
set -e
cd ulamai
python3 -m ulam prove examples/Smoke.lean \
  --theorem irrational_sqrt_two_smoke \
  --llm mock --lean mock


## 3) Prove to `.tex`: Infinitely Many Primes

In [ ]:
%%bash
set -e
cd ulamai
python3 -m ulam prove \
  --theorem infinitely_many_primes \
  --output-format tex \
  --statement "$(cat examples/ProveTexPrimes.txt)" \
  --llm mock \
  --tex-rounds 2 \
  --tex-worker-drafts 1 \
  --tex-judge-repairs 1 \
  --tex-replan-passes 1 \
  --tex-artifacts-dir runs/prove_tex


In [ ]:
%%bash
cd ulamai
echo "Generated proofs:"
ls -lah proofs || true
echo
echo "TeX artifacts root:"
ls -lah runs/prove_tex || true


## 4) Formalize from `.tex`: Polish Olympiad Problem

In [ ]:
%%bash
set -e
cd ulamai
python3 -m ulam formalize examples/FormalizePolishOlympiad.tex \
  --out examples/FormalizePolishOlympiad.lean \
  --proof-backend llm \
  --lean-backend dojo \
  --max-rounds 2 \
  --max-proof-rounds 1 \
  --no-equivalence \
  --no-llm-check \
  --artifacts-dir runs/formalize_olympiad


In [ ]:
%%bash
cd ulamai
echo "Formalized Lean preview:"
sed -n '1,120p' examples/FormalizePolishOlympiad.lean
echo
echo "Formalization artifacts:"
ls -lah runs/formalize_olympiad || true


## 5) Switch to a Real LLM Provider

For meaningful theorem-level outputs, update `.ulam/config.json` to a real provider (OpenAI/Claude/Gemini/Ollama/Codex CLI), then rerun sections 3 and 4.

Useful source files:
- `examples/ProveTexPrimes.txt`
- `examples/FormalizePolishOlympiad.tex`
- `examples/UlamAI_Prover_Tutorial.md`
